### MLFlow

In [26]:
# !pip install mlflow  # à décommenter si MLflow n'est pas installé

import os, tempfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.optim as optim
from torchvision import models
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
import mlflow
import mlflow.pytorch
import pandas as pd

device = torch.device('cuda') # Ne pas mettre de if car sinon ne fonctionne pas (pourquoi, je ne sais pas)
torch.backends.cudnn.benchmark = True
print(device)

cuda


In [28]:
df = pd.read_pickle('Projet/data_preprocess.pkl')
print(df.head())
print(df.shape)

le = LabelEncoder()
y = le.fit_transform(df['Class'])  # Convertit en entiers
y = y.astype(np.int64) 
# 1. Extraire les données du DataFrame
X = np.stack(df['img_preprocess'].values)  # Forme attendue : (N, C, H, W) ou (N, H, W, C)

# 2. Vérifier la forme de X et ajuster si nécessaire (ex: passer de (N, H, W, C) à (N, C, H, W))
if X.shape[-1] == 3:  # Si les images sont en (H, W, C)
    X = np.transpose(X, (0, 3, 1, 2))  # Passer à (N, C, H, W)

# 3. Convertir en tenseurs PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)  # float32 pour les images
y_tensor = torch.tensor(y, dtype=torch.long)    # long pour les classes

# 4. Créer le Dataset
full_dataset = TensorDataset(X_tensor, y_tensor)

# 5. Séparer en train/val (160 train, 40 val)
generator = torch.Generator().manual_seed(42)  # Pour la reproductibilité
train_set, val_set = random_split(full_dataset, [683, 171], generator=generator)

# 6. Créer les DataLoader
batch_size = 32
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)  # Pas de shuffle pour la validation

# 7. Vérification rapide
print("Nombre de lots dans train_loader :", len(train_loader))
print("Nombre de lots dans val_loader :", len(val_loader))

       Class                                     img_preprocess
0  Abrasions  [[[tensor(0.5529), tensor(0.5373), tensor(0.55...
1  Abrasions  [[[tensor(0.2000), tensor(0.2000), tensor(0.20...
2  Abrasions  [[[tensor(0.2863), tensor(0.2863), tensor(0.28...
3  Abrasions  [[[tensor(0.6078), tensor(0.6157), tensor(0.62...
4  Abrasions  [[[tensor(0.4196), tensor(0.4039), tensor(0.37...
(854, 2)
Nombre de lots dans train_loader : 22
Nombre de lots dans val_loader : 6


In [ ]:
# Toutes les expériences seront stockées dans ./mlruns (consultable via `mlflow ui`)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.chdir(r"C:\Users\USER\Desktop\Master_2_Data_Science\Deep_learning")
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("classification_plaies")

class_names = ["Abrasions","Bruises","Burns","Cut","Ingrown_nails","Laceration","Stab_wound"]
NUM_CLASSES = len(class_names)
print("Device :", device, "| Nb classes :", NUM_CLASSES)

2026/06/30 14:22:31 INFO mlflow.tracking.fluent: Experiment with name 'classification_plaies' does not exist. Creating a new experiment.


Device : cuda | Nb classes : 7


In [ ]:
def build_model(architecture, num_classes=NUM_CLASSES, dropout=0.2, fine_tune=False):

    if architecture == "resnet50":
        model = models.resnet50(pretrained=True)
        for p in model.parameters():
            p.requires_grad = fine_tune
        in_features = model.fc.in_features
        model.fc = torch.nn.Sequential(
            torch.nn.Dropout(dropout),
            torch.nn.Linear(in_features, num_classes),
        )
    elif architecture == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=True)
        for p in model.parameters():
            p.requires_grad = fine_tune
        in_features = model.classifier[1].in_features
        model.classifier = torch.nn.Sequential(
            torch.nn.Dropout(dropout),
            torch.nn.Linear(in_features, num_classes),
        )
    else:
        raise ValueError(f"Architecture inconnue : {architecture}")
    return model.to(device)


@torch.no_grad()
def evaluate_model(model, loader):
    """Renvoie un dict de métriques (recall macro = métrique cible) + la figure
    de la matrice de confusion."""
    model.eval()
    y_true, y_pred = [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    metrics = {
        "accuracy":        accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro":    recall,
        "f1_macro":        f1,
    }
    # recall par classe (utile pour repérer une classe mal détectée)
    _, rec_per_class, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(NUM_CLASSES), zero_division=0
    )
    for name, r in zip(class_names, rec_per_class):
        metrics[f"recall_{name}"] = float(r)

    # Matrice de confusion (figure pour MLflow)
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title("Matrice de confusion")
    ax.set_xlabel("Prédit"); ax.set_ylabel("Vrai")
    plt.tight_layout()
    return metrics, fig

In [ ]:
# ============================================================
# Entraînement avec logging des courbes (train/val loss) par epoch
# Doit être appelée À L'INTÉRIEUR d'un run MLflow actif.
# ============================================================

def train_model(model, train_loader, val_loader, epochs, lr, weight_decay=0.0):
    params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.Adam(params, lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=3)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        # --- train ---
        model.train()
        running = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            running += loss.item() * inputs.size(0)
        train_loss = running / len(train_loader.dataset)

        # --- validation ---
        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                val_running += criterion(model(inputs), labels).item() * inputs.size(0)
        val_loss = val_running / len(val_loader.dataset)
        scheduler.step(val_loss)

        # courbes consultables dans MLflow (onglet Metrics)
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss}, step=epoch)
        print(f"  epoch {epoch+1:02d}/{epochs} | train={train_loss:.4f} | val={val_loss:.4f}")

    return model, val_loss

In [ ]:
# ============================================================
# GRILLE EXPÉRIMENTALE : 2 architectures × 2 configs = 4 runs
# ============================================================

EPOCHS = 10

architectures = ["resnet50", "efficientnet_b0"]
configs = [
    {"name": "lr1e-3_dp0.2", "lr": 1e-3, "dropout": 0.2, "weight_decay": 0.0},
    {"name": "lr1e-4_dp0.5", "lr": 1e-4, "dropout": 0.5, "weight_decay": 1e-4},
]

for arch in architectures:
    for cfg in configs:
        run_name = f"{arch}__{cfg['name']}"
        print(f"\n=== RUN : {run_name} ===")

        with mlflow.start_run(run_name=run_name):
            # 1) Hyperparamètres
            mlflow.log_params({
                "architecture":  arch,
                "learning_rate": cfg["lr"],
                "dropout":       cfg["dropout"],
                "weight_decay":  cfg["weight_decay"],
                "batch_size":    train_loader.batch_size,
                "epochs":        EPOCHS,
                "optimizer":     "Adam",
                "transfer_learning": True,
                "backbone_frozen":   True,
            })
            mlflow.set_tag("architecture", arch)

            # 2) Construction + entraînement
            model = build_model(arch, dropout=cfg["dropout"])
            model, final_val_loss = train_model(
                model, train_loader, val_loader,
                epochs=EPOCHS, lr=cfg["lr"], weight_decay=cfg["weight_decay"],
            )

            # 3) Évaluation
            metrics, cm_fig = evaluate_model(model, val_loader)
            metrics["final_val_loss"] = final_val_loss
            mlflow.log_metrics(metrics)

            # 4) Artefacts : matrice de confusion + modèle
            mlflow.log_figure(cm_fig, "confusion_matrix.png")
            plt.close(cm_fig)
            mlflow.pytorch.log_model(model, artifact_path="model", serialization_format='pickle')  # shape (batch, C, H, W)


            print(f"  -> recall_macro={metrics['recall_macro']:.4f} "
                  f"| accuracy={metrics['accuracy']:.4f}")

print("\nTerminé. Lancez l'interface avec :  mlflow ui  (puis http://localhost:5000)")

c:\Users\USER\Desktop\Master_2_Data_Science\Deep_learning\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\USER\Desktop\Master_2_Data_Science\Deep_learning\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



=== RUN : resnet50__lr1e-3_dp0.2 ===
  epoch 01/10 | train=1.8112 | val=1.4091
  epoch 02/10 | train=1.1965 | val=1.1056
  epoch 03/10 | train=0.9795 | val=0.8886
  epoch 04/10 | train=0.8388 | val=0.7955
  epoch 05/10 | train=0.7108 | val=0.7225
  epoch 06/10 | train=0.6077 | val=0.6561
  epoch 07/10 | train=0.5518 | val=0.6265
  epoch 08/10 | train=0.5680 | val=0.6152
  epoch 09/10 | train=0.5016 | val=0.6610
  epoch 10/10 | train=0.4446 | val=0.5380


2026/06/30 14:23:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 14:23:23 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/30 14:23:23 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/30 14:23:31 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version l

  -> recall_macro=0.8636 | accuracy=0.8713

=== RUN : resnet50__lr1e-4_dp0.5 ===
  epoch 01/10 | train=1.9754 | val=1.8942
  epoch 02/10 | train=1.9238 | val=1.8239
  epoch 03/10 | train=1.8341 | val=1.7588
  epoch 04/10 | train=1.7819 | val=1.7019
  epoch 05/10 | train=1.7075 | val=1.6459
  epoch 06/10 | train=1.6404 | val=1.5931
  epoch 07/10 | train=1.6067 | val=1.5478
  epoch 08/10 | train=1.5438 | val=1.4931
  epoch 09/10 | train=1.4833 | val=1.4567
  epoch 10/10 | train=1.4363 | val=1.4078


2026/06/30 14:24:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 14:24:25 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/30 14:24:25 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/30 14:24:32 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version l

  -> recall_macro=0.7208 | accuracy=0.7135

=== RUN : efficientnet_b0__lr1e-3_dp0.2 ===
  epoch 01/10 | train=1.7165 | val=1.4285
  epoch 02/10 | train=1.2409 | val=1.1048
  epoch 03/10 | train=0.9895 | val=0.9222
  epoch 04/10 | train=0.8242 | val=0.8162
  epoch 05/10 | train=0.7128 | val=0.7389
  epoch 06/10 | train=0.6304 | val=0.6692
  epoch 07/10 | train=0.5568 | val=0.6218
  epoch 08/10 | train=0.5213 | val=0.6029
  epoch 09/10 | train=0.4725 | val=0.5665
  epoch 10/10 | train=0.4459 | val=0.5410


2026/06/30 14:24:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 14:24:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/30 14:24:55 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/30 14:25:02 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version l

  -> recall_macro=0.8350 | accuracy=0.8304

=== RUN : efficientnet_b0__lr1e-4_dp0.5 ===
  epoch 01/10 | train=1.9352 | val=1.8865
  epoch 02/10 | train=1.8940 | val=1.8299
  epoch 03/10 | train=1.8360 | val=1.7770
  epoch 04/10 | train=1.7704 | val=1.7298
  epoch 05/10 | train=1.7433 | val=1.6880
  epoch 06/10 | train=1.6738 | val=1.6417
  epoch 07/10 | train=1.6611 | val=1.5942
  epoch 08/10 | train=1.5889 | val=1.5569
  epoch 09/10 | train=1.5430 | val=1.5229
  epoch 10/10 | train=1.5269 | val=1.4847


2026/06/30 14:25:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 14:25:25 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/30 14:25:25 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/30 14:25:32 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version l

  -> recall_macro=0.6305 | accuracy=0.6316

Terminé. Lancez l'interface avec :  mlflow ui  (puis http://localhost:5000)


In [ ]:
# ============================================================
# Identification du MEILLEUR modèle (recall_macro le plus élevé)
# ============================================================
import pandas as pd

exp = mlflow.get_experiment_by_name("classification_plaies")
runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.recall_macro DESC"],
)

cols = ["tags.mlflow.runName", "params.architecture", "params.learning_rate",
        "params.dropout", "metrics.recall_macro", "metrics.accuracy",
        "metrics.f1_macro", "metrics.precision_macro"]
comparatif = runs[cols].rename(columns=lambda c: c.split(".")[-1])
print("Comparatif des runs (trié par recall_macro) :")
print(comparatif.to_string(index=False))

best = runs.iloc[0]
print(f"\nMeilleur modèle : {best['tags.mlflow.runName']}")
print(f"  recall_macro = {best['metrics.recall_macro']:.4f}")
print(f"  run_id       = {best['run_id']}")

# On le tague comme "best_model" pour le retrouver dans l'UI
with mlflow.start_run(run_id=best["run_id"]):
    mlflow.set_tag("best_model", "true")

# Rechargement du meilleur modèle pour réutilisation/inférence
best_model = mlflow.pytorch.load_model(f"runs:/{best['run_id']}/model")

Comparatif des runs (trié par recall_macro) :
                      runName    architecture learning_rate dropout  recall_macro  accuracy  f1_macro  precision_macro
       resnet50__lr1e-3_dp0.2        resnet50         0.001     0.2      0.863636  0.871345  0.863335         0.868827
efficientnet_b0__lr1e-3_dp0.2 efficientnet_b0         0.001     0.2      0.834954  0.830409  0.825906         0.828406
       resnet50__lr1e-4_dp0.5        resnet50        0.0001     0.5      0.720830  0.713450  0.699478         0.715216
efficientnet_b0__lr1e-4_dp0.5 efficientnet_b0        0.0001     0.5      0.630489  0.631579  0.608391         0.655814

Meilleur modèle : resnet50__lr1e-3_dp0.2
  recall_macro = 0.8636
  run_id       = fda73bea94a944f080f91d7c1fc19531


### Visualiser et comparer les runs dans MLflow

Depuis un terminal, à la racine du projet (là où se trouve le dossier `mlruns`) :

```bash
mlflow ui
```

Puis ouvrir **http://localhost:5000** :
1. Cliquer sur l'expérience **`classification_plaies`**.
2. Cocher les **4 runs**, cliquer sur **Compare** → c'est la capture d'écran attendue (livrable n°2).
3. Trier la colonne `recall_macro` (ordre décroissant) : le run en tête, taggé `best_model = true`, est le meilleur modèle (livrable n°3).
4. Chaque run contient ses hyperparamètres, ses métriques, la courbe `val_loss`, la matrice de confusion (`confusion_matrix.png`) et le modèle sérialisé.

### Question sur MLFlow

**Comment MLflow a-t-il aidé à structurer la démarche expérimentale ?**

MLflow a transformé une série d'essais manuels en une démarche expérimentale traçable et reproductible. Plutôt que de modifier une variable (`Model_choices`) et de réécraser à chaque fois les résultats, chaque combinaison *architecture × hyperparamètres* est devenue un **run isolé et horodaté**, enregistrant automatiquement ses paramètres, ses métriques, ses courbes d'apprentissage, sa matrice de confusion et le modèle entraîné. Trois apports concrets :

- **Comparaison objective** : l'onglet *Compare* met les 4 runs côte à côte sur une métrique unique de décision — ici le **recall macro**, choisi parce que le contexte médical impose de minimiser les faux négatifs. Le meilleur modèle est désigné par les chiffres, pas par intuition.
- **Reproductibilité** : tous les hyperparamètres (learning rate, dropout, batch size, weight decay…) sont liés à leurs résultats. On peut reconstruire ou recharger n'importe quel modèle à partir de son `run_id`, sans ambiguïté sur la configuration utilisée.
- **Structuration du code** : pour journaliser proprement, il a fallu factoriser le pipeline en fonctions claires (`build_model`, `train_model`, `evaluate_model`) et une boucle de grille, ce qui a éliminé la duplication entre les deux architectures.

**Qu'aurions-nous fait différemment sans cet outil ?**

Sans MLflow, le suivi aurait reposé sur des moyens fragiles et coûteux en temps : noter les scores dans un tableur ou un fichier texte à la main, nommer les fichiers de poids à la convention (`resnet_lr1e3.pth`…) au risque d'erreurs et d'écrasements, et conserver les matrices de confusion dans des dossiers séparés. La comparaison se serait faite *a posteriori* en rassemblant des résultats dispersés, avec un risque réel de **perte de traçabilité** (quel hyperparamètre a produit quel score ?) et de résultats non reproductibles. On aurait probablement testé moins de configurations, et la sélection du meilleur modèle aurait été plus subjective et moins défendable.